# Processamento de imagens — questões 1 e 2


## Questão 1 (`question01.py`)


In [ ]:
import cv2 as cv
import numpy as np

OUTPUT_DIR = 'answer01/'


### Função `TAM2`


A função `TAM2` amplia a imagem pelo fator indicado em `scale`, replicando os pixels da imagem original na imagem de saída. O resultado é salvo em `answer01/tam2.png`.


In [ ]:
def TAM2(path: str, scale: int):

    img = cv.imread(path)
    assert img is not None, "file could not be read, check with os.path.exists()"

    H, W, _ = img.shape

    new_H = H * scale
    new_W = W * scale
    out = np.zeros((new_H, new_W, 3), dtype=img.dtype)

    for i in range(new_H):
        for j in range(new_W):
            source_x = j * W // new_W
            source_y = i * H // new_H

            out[i][j] = img[source_y][source_x]

    cv.imwrite(f'{OUTPUT_DIR}tam2.png', out)


### Função `TAMM`


A função `TAMM` dobra a largura e a altura da imagem. Primeiro posiciona os pixels originais e, em seguida, calcula médias entre pixels vizinhos para preencher as posições intermediárias. Salva imagens das etapas e o resultado final em `answer01/`.


In [ ]:
def TAMM(path: str):

    img = cv.imread(path)
    assert img is not None, "file could not be read, check with os.path.exists()"

    H, W, _ = img.shape
    new_H = H * 2
    new_W = W * 2
    out = np.zeros((new_H, new_W, 3), dtype=img.dtype)

    for i in range(0,new_H, 2):
        for j in range(0,new_W,2):
            out[i][j] = img[i//2][j//2]
    cv.imwrite(f'{OUTPUT_DIR}tamm_step1.png', out)

    # linhas 
    for i in range(0,new_H,2):
        for j in range(1,new_W,2):
            if (j + 1) >= new_W:
                out[i][j] = img[i//2][j//2]
                continue
            prev = img[i//2][(j//2)]
            successor = img[i//2][(j//2) + 1]
            mean = (prev.astype(np.uint16) + successor.astype(np.uint16) + 1) // 2

            out[i][j] = mean.astype(np.uint8)   
    cv.imwrite(f'{OUTPUT_DIR}tamm_step2.png', out)

    # colunas
    for i in range(1,new_H,2):
        for j in range(0,new_W):
            if (i + 1) >= new_H:
                out[i][j] = out[i-1][j]
                continue
            prev = out[i-1][j]
            successor = out[i+1][j]
            mean = (prev.astype(np.uint16) + successor.astype(np.uint16) + 1) // 2

            out[i][j] = mean.astype(np.uint8)   
            
    cv.imwrite(f'{OUTPUT_DIR}tamm_step3.png', out)


### Função `SURPERRES`


A função `SURPERRES` combina duas imagens de mesma dimensão em uma imagem com o dobro da largura e da altura, usando cada imagem para preencher posições alternadas. O resultado é salvo em `answer01/superres.png`.


In [ ]:
def SURPERRES(path1: str, path2: str):
    img1 = cv.imread(path1)
    assert img1 is not None, "img1 could not be read, check with os.path.exists()"
    img2 = cv.imread(path2)
    assert img2 is not None, "img2 could not be read, check with os.path.exists()"

    assert img1.shape == img2.shape, "images are not compatible"

    H, W, _ = img1.shape
    new_H = H*2
    new_W = W*2
    out = np.zeros((new_H, new_W, 3), dtype=img1.dtype)

    for i in range(new_H):
        for j in range(new_W):
            if i % 2 == 0 and j % 2 == 0:
                out[i][j] = img1[i//2][j//2]
            if i % 2 == 1 and j % 2 == 1:
                out[i][j] = img2[i//2][j//2]

    cv.imwrite(f"{OUTPUT_DIR}superres.png", out)


### Execução principal


In [ ]:
if __name__ == "__main__":


    path = "images/fruit01.png"
    path2 = "images/fruit02.png"
    TAM2(path, 8)
    TAMM(path)
    SURPERRES(path, path2)


## Questão 2 (`question02.py`)


In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR = "answer02"

def power_law(path: str, gamma: float, c: float = 1):

    img = cv.imread(path, flags=cv.IMREAD_GRAYSCALE)
    assert img is not None, "file could not be read, check with os.path.exists()"

    out = np.zeros(img.shape)
    cv.pow(img/255, gamma, out)
    out = np.clip(np.round(out * c * 255), 0, 255).astype(np.uint8)

    cv.imwrite(f"{OUT_DIR}/power-law.png", out)


def hist_cdf(im: np.ndarray):
    """Retorna histograma e CDF normalizada [0, 1] de uma imagem 8 bits de 1 canal."""
    hist = np.bincount(im.ravel(), minlength=256)
    cdf = hist.cumsum()
    return hist, cdf / cdf[-1]


def salvar_hist(hist: np.ndarray, titulo: str, caminho: str) -> None:
    plt.figure(figsize=(7, 4))
    plt.bar(range(256), hist, width=1)
    plt.title(titulo)
    plt.xlabel("Intensidade")
    plt.ylabel("Nº de pixels")
    plt.xlim(0, 255)
    plt.tight_layout()
    plt.savefig(caminho, dpi=150)
    plt.close()


def salvar_cdf(cdf: np.ndarray, titulo: str, caminho: str) -> None:
    plt.figure(figsize=(7, 4))
    plt.plot(cdf)
    plt.title(titulo)
    plt.xlabel("Intensidade")
    plt.ylabel("CDF")
    plt.xlim(0, 255)
    plt.ylim(0, 1.02)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(caminho, dpi=150)
    plt.close()


def equalizar(path: str) -> None:
    
    img = cv.imread(path, flags=cv.IMREAD_GRAYSCALE)
    assert img is not None, "file could not be read, check with os.path.exists()"
    
    
    eq = cv.equalizeHist(img)
 
    h1, c1 = hist_cdf(img)
    h2, c2 = hist_cdf(eq)
 
    salvar_hist(h1, "Histograma - original", f"{OUT_DIR}/hist_original.png")
    salvar_cdf(c1, "CDF - original", f"{OUT_DIR}/cdf_original.png")
    salvar_hist(h2, "Histograma - equalizado", f"{OUT_DIR}/hist_equalizado.png")
    salvar_cdf(c2, "CDF - equalizada", f"{OUT_DIR}/cdf_equalizada.png")
    cv.imwrite(f"{OUT_DIR}/imagem_equalizada.png", eq)
    
if __name__ == '__main__':
    power_law("images/car.png", 0.5, 1)
    equalizar("images/car.png")
